In [6]:
# Required Python packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
from scipy.stats import norm
import copy
import os

#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import find_last_thurs_date_of_month, date_of_prev_thurs, \
                                                     prev_weekday, prev_workday_if_holiday, \
                                                     next_working_day_incl_input_date, next_weekday, \
                                                     next_weekday_incl_input_date, \
                                                     get_all_monthly_option_expiries, find_wkly_expries \
                                                     

#Import pricing functions
from ipynb.fs.full.user_defined_vik_functions import BSM_call, BSM_put

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data, load_all_wkly_data_of_the_month, find_latest_available_mibor

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Path Settings
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path


In [7]:
#Index of interest
stock_ident = "BANKNIFTY"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
# periods_of_interest = {2019: [11, 12], 2020: [1, 2, 3]}
periods_of_interest = {2020: [3]}
#List oh holidays
holidays_list = [date(2019, 10, 28), date(2019, 11, 12), date(2019, 12, 25), date(2020, 2, 21), date(2020, 3, 10)]


In [8]:
# The following functions are user-defined functions used for 
# generating implied vol surface using Brent's Alogrithm

#Error Function - For European call and European put options
#Input: mkt_price of the option, option_type: "CE" for European Call, "PE" for European Put
#      (s_t, k, r, dt) -> (current stock price, strike, risk-free rate, time to maturity)
#Output: Error of model price in comparison with mkt price
def error_func(mkt_price, s_t, k, r, dt, option_type, iv):
    if (option_type == "CE"):
        model_price = BSM_call(s_t, k, r, iv, dt)
    else:
        model_price = BSM_put(s_t, k, r, iv, dt)
    error = model_price - mkt_price
    return error

#### O(n) Algorithm to find product of other elements ####
# Input: a list
#Output: A list with product of other elements except the element in the index
def product_other_elem(arr):
    n = len(arr)
    prod_arr = list(np.ones(n).astype(np.float64))
    
    temp = 1.0
    for i in range(n):
        prod_arr[i] = temp
        temp = temp * arr[i]
    temp = 1.0
    for i in range(n-1, -1, -1):
        prod_arr[i] = prod_arr[i] * temp
        temp = temp * arr[i]
    return prod_arr

#Generates the iv(k) as function of iv(k-1), iv(k-2), iv(k-3), mkt_price, BS price
#### Using Brent's Algorithm ####

#Inverse Quadratic interpolation
def inverse_quadratic_iv(mkt_price, s_t, k, r, dt, option_type, iv_list):
    g = lambda iv: error_func(mkt_price, s_t, k, r, dt, option_type, iv)
    g_k = [g(iv) for iv in iv_list]
    g_num = product_other_elem(g_k)
    g_den = [np.prod(g_k[i] - np.array(g_k[:i] + g_k[i+1:])) for i in range(0, len(iv_list))]
    next_iv = np.sum(np.array([(iv_list[i] * g_num[i]) / g_den[i] for i in range(0, len(iv_list))]))
    return next_iv

#Secant method
def secant_method_iv(mkt_price, s_t, k, r, dt, option_type, iv_list):
    g = lambda iv: error_func(mkt_price, s_t, k, r, dt, option_type, iv)
    g_k = [g(iv) for iv in iv_list]
    next_iv = (iv_list[1] - g_k[1]) * ((iv_list[1] - iv_list[0])/(g_k[1]-g_k[0]))
    return next_iv

#Bisection method
def bisection_method_iv(mkt_price, s_t, k, r, dt, option_type, iv_list):
    next_iv = (iv_list[0] + iv_list[1])/2
    return next_iv

#Swap the elements of 2 elements list
def swap(iv_list):
    temp = iv_list[1]
    iv_list[1] = iv_list[0]
    iv_list[0] = temp
    return iv_list

#Generate IV using Brent's algorithm
def gen_brents_iv(mkt_price, s_t, k, r, dt, option_type, iv_list_initial):
    iv_list = copy.deepcopy(iv_list_initial)
    g = lambda iv: error_func(mkt_price, s_t, k, r, dt, option_type, iv)
    
#     print(g(iv_list[0]), g(iv_list[1]))
    
    try:
        assert (g(iv_list[0]) * g(iv_list[1])) < 0
    except AssertionError:
        print("BSMVol0_price - mkt_price, BSMVol1_price - mkt_price: ", g(iv_list[0]), g(iv_list[1]))
        print("Trade_terms: s_t, k, dt, option_type, iv_list: (", s_t, k, dt, option_type, iv_list, ")")
        print("Mkt Price: ", mkt_price)
        if (option_type == "CE"):
            print("IV0 BSM price: ", BSM_call(s_t, k, r, iv_list[0], dt))
            print("IV1 BSM price: ", BSM_call(s_t, k, r, iv_list[1], dt))
        else:
            print("IV0 BSM price: ", BSM_call(s_t, k, r, iv_list[0], dt))
            print("IV1 BSM price: ", BSM_call(s_t, k, r, iv_list[1], dt))
        print("The initialisations doesn't contain the solution within for interpolation")
    
    if (abs(iv_list[0] ) < abs(iv_list[1])):
        iv_list = swap(iv_list)

    mflag = 1
    
    opt_iv = iv_list[1]    
    num_iter = 0 #maximum iteration is set as 1000 in the while loop below
    eps = 0.0001 # 1 basis point is set as the maximum error allowed for vol

#     print("Starting Error: ", g(opt_iv))
#    error is difference of market and model price
    
    while (abs(g(opt_iv)) >= 0.000001 and num_iter <=1000):   
        if (g(iv_list[0]) != g(iv_list[2]) and g(iv_list[1]) != g(iv_list[2])):
            #Inverse Quadratic Interpolation
            opt_iv = inverse_quadratic_iv(mkt_price, s_t, k, r, dt, option_type, iv_list)
        else:
            #Secant Method
            opt_iv = secant_method_iv(mkt_price, s_t, k, r, dt, option_type, iv_list)
        
        if ((( 3 * iv_list[0] + iv_list[1])/4 < opt_iv < iv_list[1]) \
            or ( mflag == 1 and (abs(opt_iv - iv_list[1]) >= abs(iv_list[1] - iv_list[2])/2 )) \
            or ( mflag == 0 and (abs(opt_iv - iv_list[1]) >= abs(iv_list[2] - prev_iv_2)/2 )) \
            or (mflag == 1 and (abs(iv_list[1] - iv_list[2]) < abs(eps))) \
            or (mflag == 0 and (abs(iv_list[2] - prev_iv_2) < abs(eps)))):
            
            opt_iv = (iv_list[0] + iv_list[1]) / 2
            mflag = 1
        else:
            mflag = 0
            
#         print("Iteration",num_iter, ": ", "Price Difference is ", g(opt_iv))
#         print("Model IV: ",  opt_iv)

        prev_iv_2 = iv_list[2]
        iv_list[2] = iv_list[1]
        
        if (g(iv_list[0]) * g(opt_iv) < 0):
            iv_list[1] = opt_iv
        else:
            iv_list[0] = opt_iv
        
        if (abs(iv_list[0] ) < abs(iv_list[1])):
            iv_list = swap(iv_list)
        
        num_iter = num_iter + 1
    return opt_iv

#Function to generate Implied volatility Surface based on Brent's Algorithm
#Note: In vol surface: x-axis is moneyness ie. S/K for call options and K/S for put options
#                      y-axis is Time to maturity
#                      z-axis is implied volatility
#Inputs:
#       dt_list = [tenor1, tenor2....]
#       price_lists_tenorwise = [[Price for all options in tenor 1],[for tenor2],....]
#       strike_lists_tenorwise = [[Strike for all options in tenor 1],[for tenor2],....]
#       s_t - current stock price, r - risk free rate, option_type is "CE" or "PE"
#      initialisation of brent's iv is: [0.0000000001, 50, 50], 
#                 S.T, iv[0] creates a +ve error and iv[1] creates negative error,
#                where, error = (model price - mkt_price)
#Output:
#      A list of iv surface: [dict for tenor 1, dict for tenor 2,...]
#                      dict = {"Moneyness:" [], "T": [list of same value dt], "Impl_Vol": []}
def gen_brent_iv_surface(price_lists_tenorwise, s_t, strike_lists_tenorwise, r, dt_list, option_type, iv_list=[0.0000000001, 50, 50]):
    iv_surface = []
    for i in range(0, len(dt_list)):
        dt = dt_list[i]
        strike_list = strike_lists_tenorwise[i]
        price_list = price_lists_tenorwise[i]
        if(option_type == "CE"):
            moneyness_list = [s_t/strike for strike in strike_list]
        else:
            moneyness_list = [strike/s_t for strike in strike_list]
        dt_list_curr_tenor = [dt for evry_elm in price_list]
        spot_list = [s_t for evry_elm in price_list]
        impl_vol_list = [gen_brents_iv(price_list[i], s_t, strike_list[i], r, dt, option_type, iv_list) for i in range(0, len(price_list))]
        
        vol_smile_dict = {"Moneyness":moneyness_list,"T":dt_list_curr_tenor, "Spot": spot_list, "Strike":strike_list, "Impl_Vol":impl_vol_list}
        iv_surface.append(vol_smile_dict)
    return iv_surface

#Function to generate iv_surface dataframe from (iv-surface list -> list of iv_smile dictionaries)
def gen_iv_surf_dataframe(iv_surface_list_of_dict):
    list_of_df = [pd.DataFrame(iv_surf_dict) for iv_surf_dict in iv_surface_list_of_dict]
    list_of_df = [iv_df.sort_values('Moneyness') for iv_df in list_of_df]
    df_iv_surface = pd.concat(list_of_df, axis=0)
    return df_iv_surface

# # We generate and store the volatility surface for 26th March expiry.
# # Each day, we generate a volatility surface with time to maturity = 26th march - current date
# # ACT/365 day count convention is used
# #Input: periods_of =_interest as discussed above, holidays list, product type - "CE" or "PE"
# #Output: List of vol surface dataframes and saved as csv in the data location
# def gen_daily_vol_smiles_for_fixed_expiries(periods_of_interest, holidays_list, input_path, output_path, stock_ident, prod_type = "CE"):
#     expiries_of_interest = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
#     #Product Type corresponds to product type we are hedging - theoretically vol smile is same for call and put
#     # Put call parity to be tested and volatility arbitrage to be tested at all dates - as analysis of this exercise
#     df_mkt = load_all_mthly_data(expiries_of_interest, input_path, prod_type_lists=["CE", "PE"], stock_ident=stock_ident)

#     df_futures_mkt = load_all_mthly_data(expiries_of_interest, input_path, prod_type_lists=["FUT"], stock_ident=stock_ident)
# #     df_mibor_hist = pd.read_csv(relative_path + "mibor.csv")
    
#     mthly_list_of_vol_smiles = {}
#     for expiry_date in expiries_of_interest:
#         fixed_expiry_volsmiles_list = []
#         start_date = date_of_prev_thurs(expiry_date, 4)
#         curr_date =start_date
#         df = df_mkt[get_mthly_df_name_from_expiry(expiry_date, prod_type, holidays_list)]
#         df_futures = df_futures_mkt[get_mthly_df_name_from_expiry(expiry_date, "FUT", holidays_list)]
        
#         while(curr_date < expiry_date):
#             dt = float((expiry_date - curr_date).days) / 365
#             df_curr = df[df['Date'] == curr_date.strftime("%d-%b-%Y")]
#             s_t = float(df_curr['Underlying Value'].tolist()[0])
#             price_lists_tenorwise = [df_curr['Settle Price'].tolist()]
#             strike_lists_tenorwise = [df_curr['Strike Price'].tolist()]
#             dt_list = np.array([dt])
            
# #We had risk free interest rate from futures 
#             df_curr_futures = df_futures[df_futures['Date'] == curr_date.strftime("%d-%b-%Y")]
#             F_t = float(df_curr_futures['Settle Price'].tolist()[0])
#             r = float(1 / dt) * np.log(F_t / s_t)

#             # Note: The historical data fro mibor file is named as mibor.csv
# #             r = find_latest_available_mibor(df_mibor_hist, curr_date)
# # We also have another idea of generating r by implying from put-call parity
# # Temporarily, assumed r to be constant
# #             r = 0.03

#             dict_vol_smile = gen_brent_iv_surface(price_lists_tenorwise, s_t, strike_lists_tenorwise, r, dt_list, option_type=prod_type, iv_list=[0.00000000000001, 50, 50])[0]
#             dict_vol_smile['Date'] = [curr_date.strftime("%d-%b-%Y") for i in range(0, len(dict_vol_smile['Moneyness']))]
#             fixed_expiry_volsmiles_list.append(pd.DataFrame(dict_vol_smile))
#             curr_date = next_weekday(curr_date)
#             curr_date = next_working_day_incl_input_date(curr_date, holidays_list)
#         df_fixed_expiry_vol_surface = pd.concat(fixed_expiry_volsmiles_list, axis=0)
#         df_fixed_expiry_vol_surface.sort_values(by=['Date', 'Moneyness'])
#         mthly_list_of_vol_smiles["Daily_vol_smile_" + prod_type +"_" + expiry_date.strftime("%d-%b-%Y")] = df_fixed_expiry_vol_surface
#     for key in mthly_list_of_vol_smiles.keys():
#         df = pd.DataFrame(mthly_list_of_vol_smiles[key])
#         df.to_csv(output_path + "A1_" + key + ".csv", index = False)
#     return mthly_list_of_vol_smiles 


# We generate and store the volatility surface for 26th March expiry.
# Each day, we generate a volatility surface with time to maturity = 26th march - current date
# ACT/365 day count convention is used
#Input: periods_of =_interest as discussed above, holidays list, product type - "CE" or "PE"
#Output: List of vol surface dataframes and saved as csv in the data location
def gen_daily_vol_smiles_for_fixed_expiries(periods_of_interest, holidays_list, input_path, output_path, \
                                            stock_ident, prod_type = "CE"):
    expiries_of_interest = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
    #Product Type corresponds to product type we are hedging - theoretically vol smile is same for call and put
    # Put call parity to be tested and volatility arbitrage to be tested at all dates - as analysis of this exercise
    df_mkt = load_all_mthly_data(expiries_of_interest, input_path, \
                                 holidays_list, prod_type_lists=["CE", "PE"], stock_ident=stock_ident)

    df_futures_mkt = load_all_mthly_data(expiries_of_interest, input_path, \
                                         holidays_list, prod_type_lists=["FUT"], stock_ident=stock_ident)
#     df_mibor_hist = pd.read_csv(relative_path + "mibor.csv")
    
    mthly_list_of_vol_smiles = {}
    for expiry_date in expiries_of_interest:
        fixed_expiry_volsmiles_list = []
        if (expiry_date.month == 1):
            last_month = 12
            last_year = expiry_date.year - 1
        else:
            last_month = expiry_date.month - 1
            last_year = expiry_date.year 
            
        start_date = prev_workday_if_holiday(find_last_thurs_date_of_month(last_year, \
                                                                               last_month), \
                                                 holidays_list)

#         start_date = date_of_prev_thurs(expiry_date, 4)
        curr_date =start_date
        df = df_mkt[get_mthly_df_name_from_expiry(expiry_date, prod_type, holidays_list)]
        df_futures = df_futures_mkt[get_mthly_df_name_from_expiry(expiry_date, "FUT", holidays_list)]
        wkly_expiries_list = [date_of_prev_thurs(expiry_date, no_of_thurs) for no_of_thurs in range(4, 0, -1)]
        wkly_expiries_list = [prev_workday_if_holiday(date, holidays_list) for date in wkly_expiries_list]
        
        dict_weekly_dfs = load_all_wkly_data_of_the_month(expiry_date, holidays_list, input_path, prod_type_lists=["CE", "PE"], stock_ident = stock_ident)
        
        week_id=0
        while(curr_date < expiry_date):
            
            if (curr_date == wkly_expiries_list[week_id]):
#                 print(curr_date, wkly_expiries_list[week_id], week_id)
                wk_start = wkly_expiries_list[week_id]
                wk_end = prev_workday_if_holiday(wkly_expiries_list[week_id] + timedelta(days=7), holidays_list)
                
                dt = float((expiry_date - curr_date).days) / 365
                dt_wk = float((wk_end - curr_date).days) / 365
                week_file_name = "OPTIDX_" + stock_ident + "_" + prod_type + "_" \
                        + wk_start.strftime("%d-%b-%Y") +  "_TO_" \
                        + wk_end.strftime("%d-%b-%Y") + ".csv"
                df_week = dict_weekly_dfs["df" + "_" + week_file_name[:-4]]

                df_wk = df_week[df_week['Date'] == curr_date.strftime("%d-%b-%Y")]
                df_curr = df[df['Date'] == curr_date.strftime("%d-%b-%Y")]
                
                s_t = float(df_curr['Underlying Value'].tolist()[0])
            
                price_lists_tenorwise = [df_curr['Settle Price'].tolist(), df_wk['Settle Price'].tolist()]
                strike_lists_tenorwise = [df_curr['Strike Price'].tolist(), df_wk['Strike Price'].tolist()]
                dt_list = np.array([dt, dt_wk])
                week_id = week_id+1
                if (week_id >= len(wkly_expiries_list)):
                    week_id = 0
                
            else:
                dt = float((expiry_date - curr_date).days) / 365
                df_curr = df[df['Date'] == curr_date.strftime("%d-%b-%Y")]
                s_t = float(df_curr['Underlying Value'].tolist()[0])

                price_lists_tenorwise = [df_curr['Settle Price'].tolist()]
                strike_lists_tenorwise = [df_curr['Strike Price'].tolist()]
                dt_list = np.array([dt])

#We had risk free interest rate from futures 
            df_curr_futures = df_futures[df_futures['Date'] == curr_date.strftime("%d-%b-%Y")]
            F_t = float(df_curr_futures['Settle Price'].tolist()[0])
            r = float(1 / dt) * np.log(F_t / s_t)

            # Note: The historical data fro mibor file is named as mibor.csv
#             r = find_latest_available_mibor(df_mibor_hist, curr_date)
# We also have another idea of generating r by implying from put-call parity
# Temporarily, assumed r to be constant
#             r = 0.03

            list_dict_vol_smile = gen_brent_iv_surface(price_lists_tenorwise, s_t, strike_lists_tenorwise, r, dt_list, option_type=prod_type, iv_list=[0.00000000000001, 50, 50])
            dict_vol_smile = gen_iv_surf_dataframe(list_dict_vol_smile)
            dict_vol_smile['Date'] = [curr_date.strftime("%d-%b-%Y") for i in range(0, len(dict_vol_smile['Moneyness']))]
            fixed_expiry_volsmiles_list.append(pd.DataFrame(dict_vol_smile))
            curr_date = next_weekday(curr_date)
            curr_date = next_working_day_incl_input_date(curr_date, holidays_list)
        df_fixed_expiry_vol_surface = pd.concat(fixed_expiry_volsmiles_list, axis=0)
        df_fixed_expiry_vol_surface.sort_values(by=['Date', 'Moneyness'])
        mthly_list_of_vol_smiles["Daily_vol_smile_" + prod_type +"_" + expiry_date.strftime("%d-%b-%Y")] = df_fixed_expiry_vol_surface
    for key in mthly_list_of_vol_smiles.keys():
        df = pd.DataFrame(mthly_list_of_vol_smiles[key])
        df.to_csv(output_path + "A1_" + stock_ident + "_" + key + ".csv", index = False)
    return mthly_list_of_vol_smiles 


In [9]:

#Generate Daily Vol smile - Time axis for each date 
#correspond to (Expiry date of monthly option - Current date) with ACT/365 Day count convention
# gen_daily_vol_smiles_for_fixed_expiries(periods_of_interest, holidays_list=holidays_list, input_path=input_data_path, output_path=output_data_path, stock_ident=stock_ident, prod_type = "CE")
# gen_daily_vol_smiles_for_fixed_expiries(periods_of_interest, holidays_list=holidays_list, input_path=input_data_path, output_path=output_data_path, stock_ident=stock_ident, prod_type = "PE")


BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  52.60979441924155 30239.60979428065
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 33200 0.07671232876712329 PE [1e-14, 50, 50] )
Mkt Price:  2884.95
IV0 BSM price:  0.0
IV1 BSM price:  30186.999999861404
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  54.38256488436036 30241.382564745556
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 33300 0.07671232876712329 PE [1e-14, 50, 50] )
Mkt Price:  2982.95
IV0 BSM price:  0.0
IV1 BSM price:  30186.999999861197
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  55.905335349479174 30242.90533521047
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 33400 0.07671232876712329 PE [1e-14, 50, 50] )
Mkt Price:  3081.2
IV0 BSM price:  0.0
IV1 BSM price:  30186.999999860986
The initialisations doesn't contain the s

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  8.782089768060587 30195.782089631568
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 32200 0.07671232876712329 PE [1e-14, 50, 50] )
Mkt Price:  1931.05
IV0 BSM price:  0.0
IV1 BSM price:  30186.999999863507
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  16.354860233179352 30203.35486009647
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 32300 0.07671232876712329 PE [1e-14, 50, 50] )
Mkt Price:  2023.25
IV0 BSM price:  0.0
IV1 BSM price:  30186.999999863296
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  39.37763069829816 30226.377630561383
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 32400 0.07671232876712329 PE [1e-14, 50, 50] )
Mkt Price:  2100.0
IV0 BSM price:  0.0
IV1 BSM price:  30186.99999986308
The initialisations doesn't contain the s

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  16.17101860432558 30185.833650910354
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 34700 0.019178082191780823 PE [1e-14, 50, 50] )
Mkt Price:  4477.1
IV0 BSM price:  0.0
IV1 BSM price:  30169.662632306026
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  16.21416275016236 30185.85195948764
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 34800 0.019178082191780823 PE [1e-14, 50, 50] )
Mkt Price:  4577.0
IV0 BSM price:  0.0
IV1 BSM price:  30169.63779673748
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  16.25730689599186 30185.870306530567
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 34900 0.019178082191780823 PE [1e-14, 50, 50] )
Mkt Price:  4676.9
IV0 BSM price:  0.0
IV1 BSM price:  30169.612999634577
The initialisations doesn't contain the s

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  16.037568125150756 30186.128456693277
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 33000 0.019178082191780823 PE [1e-14, 50, 50] )
Mkt Price:  2778.2
IV0 BSM price:  0.0
IV1 BSM price:  30170.09088856813
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  15.430712270986987 30185.49608482798
Trade_terms: s_t, k, dt, option_type, iv_list: ( 30187.0 33100 0.019178082191780823 PE [1e-14, 50, 50] )
Mkt Price:  2878.75
IV0 BSM price:  0.0
IV1 BSM price:  30170.065372556997
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  231.8071231400886 29378.95712281295
Trade_terms: s_t, k, dt, option_type, iv_list: ( 29147.15 33200 0.07397260273972603 PE [1e-14, 50, 50] )
Mkt Price:  3927.6
IV0 BSM price:  0.0
IV1 BSM price:  29147.149999672863
The initialisations doesn't contain the

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  254.568137324075 29401.71813698629
Trade_terms: s_t, k, dt, option_type, iv_list: ( 29147.15 35400 0.07397260273972603 PE [1e-14, 50, 50] )
Mkt Price:  6111.9
IV0 BSM price:  0.0
IV1 BSM price:  29147.149999662215
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  76.17515376592792 29223.325153450314
Trade_terms: s_t, k, dt, option_type, iv_list: ( 29147.15 30900 0.07397260273972603 PE [1e-14, 50, 50] )
Mkt Price:  1775.85
IV0 BSM price:  0.0
IV1 BSM price:  29147.149999684385
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  106.41706414629152 29253.567063829658
Trade_terms: s_t, k, dt, option_type, iv_list: ( 29147.15 31100 0.07397260273972603 PE [1e-14, 50, 50] )
Mkt Price:  1946.25
IV0 BSM price:  0.0
IV1 BSM price:  29147.149999683366
The initialisations doesn't contain th

...

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  329.96613874982904 15973.62541939418
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 30600 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  13483.65
IV0 BSM price:  0.0
IV1 BSM price:  15643.659280644351
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  331.06488430129866 15972.484403464821
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 30700 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  13583.6
IV0 BSM price:  0.0
IV1 BSM price:  15641.419519163523
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  332.113629852769 15971.298080542138
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 30800 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  13683.6
IV0 BSM price:  0.0
IV1 BSM price:  15639.184450689369
The initialisations doesn't contain

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  52.442838051356375 15951.795065070954
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 20500 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  3555.25
IV0 BSM price:  0.0
IV1 BSM price:  15899.352227019597
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  221.0415836028269 16117.51148646251
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 20600 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  3487.7
IV0 BSM price:  0.0
IV1 BSM price:  15896.469902859686
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  222.34032915429725 16115.936544891349
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 20700 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  3587.45
IV0 BSM price:  0.0
IV1 BSM price:  15893.596215737052
The initialisations doesn't contain 

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  246.9127312866376 16079.352712012842
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 22900 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  5785.95
IV0 BSM price:  0.0
IV1 BSM price:  15832.439980726203
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  146.76147683810814 15976.510395540974
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 23000 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  5987.15
IV0 BSM price:  0.0
IV1 BSM price:  15829.748918702864
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  249.11022238957776 16076.175385330303
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 23100 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  5985.85
IV0 BSM price:  0.0
IV1 BSM price:  15827.065162940728
The initialisations doesn't contai

BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  273.8813700733881 16041.1321505975
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 25400 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  8285.2
IV0 BSM price:  0.0
IV1 BSM price:  15767.250780524111
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  276.0288611763299 16038.242391914175
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 25600 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  8485.15
IV0 BSM price:  0.0
IV1 BSM price:  15762.213530737843
The initialisations doesn't contain the solution within for interpolation
BSMVol0_price - mkt_price, BSMVol1_price - mkt_price:  277.1276067277995 16036.831837902146
Trade_terms: s_t, k, dt, option_type, iv_list: ( 17107.3 25700 0.005479452054794521 PE [1e-14, 50, 50] )
Mkt Price:  8585.1
IV0 BSM price:  0.0
IV1 BSM price:  15759.704231174346
The initialisations doesn't contain the 

{'Daily_vol_smile_PE_26-Mar-2020':     Moneyness         T      Spot  Strike  Impl_Vol         Date
 78   0.887799  0.076712  30187.00   26800  0.154953  27-Feb-2020
 79   0.891112  0.076712  30187.00   26900  0.155113  27-Feb-2020
 80   0.894425  0.076712  30187.00   27000  0.240697  27-Feb-2020
 81   0.897737  0.076712  30187.00   27100  0.154997  27-Feb-2020
 82   0.901050  0.076712  30187.00   27200  0.154872  27-Feb-2020
 ..        ...       ...       ...     ...       ...          ...
 19   1.893832  0.002740  18481.05   35000  4.657831  25-Mar-2020
 20   1.899243  0.002740  18481.05   35100  4.516315  25-Mar-2020
 21   1.904654  0.002740  18481.05   35200  4.536111  25-Mar-2020
 22   1.910065  0.002740  18481.05   35300  4.555853  25-Mar-2020
 23   1.915476  0.002740  18481.05   35400  4.571793  25-Mar-2020
 
 [3066 rows x 6 columns]}